In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
batch_size = 4
seq_len = 64
embed_dim = 128

x = torch.randn(batch_size, seq_len, embed_dim)
print("Input shape:", x.shape)

Input shape: torch.Size([4, 64, 128])


## Implement Attention without any learnable parameters

In [3]:
## First compute the attention scores
similarity = (x @ x.transpose(-2, -1))

print(f"Before softmax, similarity variance: {similarity.var()}") # to high variance
similarity_norm = similarity / (embed_dim ** 0.5)
print(f"After normalization, similarity variance: {similarity_norm.var()}")

print("Similarity shape:", similarity.shape)

## Now apply softmax to get the attention weights
attention_weights = F.softmax(similarity_norm, dim=-1)

## Now compute the attention output
context_vectors = attention_weights @ x
print("Context vectors shape:", context_vectors.shape)

Before softmax, similarity variance: 393.93231201171875
After normalization, similarity variance: 3.0775961875915527
Similarity shape: torch.Size([4, 64, 64])
Context vectors shape: torch.Size([4, 64, 128])


## Add Learnable Parameters

In [4]:
# Create a linear layer that projects from 10 dimensions to 20 dimensions
linear = nn.Linear(10, 20)

# Create a random tensor with shape (batch_size=4, sequence_length=5, feature_dim=10)
rand = torch.randn(4, 5, 10)
print("Input to linear layer shape:", rand.shape)

# Apply the linear transformation
output = linear(rand)
print("Output shape after linear layer:", output.shape)

# Detailed explanation of how linear layers work with multi-dimensional tensors:
# 1. The linear layer only operates on the LAST dimension of the input tensor
# 2. All other dimensions are preserved and treated as "batch dimensions"
# 3. Internally, PyTorch reshapes the input from (4, 5, 10) to (4*5, 10) = (20, 10)
# 4. Then applies the linear transformation: (20, 10) @ (10, 20) + bias = (20, 20)
# 5. Finally reshapes back to (4, 5, 20)
# 
# This is equivalent to:
# rand_reshaped = rand.view(-1, 10)  # Shape: (20, 10)
# output_reshaped = linear(rand_reshaped)  # Shape: (20, 20)
# output = output_reshaped.view(4, 5, 20)  # Shape: (4, 5, 20)
#
# This behavior makes linear layers very convenient for:
# - Processing sequences (batch_size, seq_len, features)
# - Processing images after flattening (batch_size, height, width, channels)
# - Any tensor where you want to transform only the feature dimension

Input to linear layer shape: torch.Size([4, 5, 10])
Output shape after linear layer: torch.Size([4, 5, 20])


In [5]:
class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.scale = embed_dim ** 0.5
        
        # create pointwise projection layers
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, x):
        # compute query, key, value
        query = self.query_proj(x)
        key = self.key_proj(x)
        value = self.value_proj(x)
        
        # compute attention scores
        similarity = (query @ key.transpose(-2, -1)) / self.scale
        
        # apply softmax to get attention weights
        attention_weights = F.softmax(similarity, dim=-1)
        
        # compute context vectors
        context_vectors = attention_weights @ value
        
        return context_vectors

In [6]:
attention = Attention(embed_dim = 128)
context_vectors = attention(x)
print("Context vectors shape with learnable parameters:", context_vectors.shape)

Context vectors shape with learnable parameters: torch.Size([4, 64, 128])


## MultiHeaded Attention


```markdown
## 🔹 Step 1: Split Embedding into Multiple Heads

We start with input shape:
```
[Batch x Seq_len x Embed_dim]
```

Assuming `Embed_dim = Num_Heads × Head_Dim`,  
we reshape into:
```
[Batch x Seq_len x Num_Heads x Head_Dim]
```

> Example: `[B x 8 x 9]` → `[B x 8 x 3 x 3]` if `Num_Heads = 3`, `Head_Dim = 3`

---

## 🔹 Step 2: Transpose for Parallel Attention

To perform attention across heads, transpose:
```
[Batch x Seq_len x Num_Heads x Head_Dim] →
[Batch x Num_Heads x Seq_len x Head_Dim]
```

This enables **parallel attention computation** across heads.

---

## 🔹 Step 3: Scaled Dot-Product Attention

We compute attention scores:
```
Q @ Kᵀ → [Batch x Num_Heads x Seq_len x Seq_len]
```

- Q shape: `[B x H x S x D]`  
- Kᵀ shape: `[B x H x D x S]`

Result: Attention matrix for each head and sample.

✅ Attention is now **parallelized per head and per sample**.

---

## 🔹 Step 4: Scale & Softmax

Scale attention scores by:
```
1 / sqrt(Head_Dim)
```

Then apply `softmax` **along last dimension** (per row):
```
[Batch x Num_Heads x Seq_len x Seq_len]
```

---

## 🔹 Step 5: Multiply by Values

Values originally shaped:
```
[Batch x Seq_len x Num_Heads x Head_Dim]
```

Transpose to:
```
[Batch x Num_Heads x Seq_len x Head_Dim]
```

Now compute:
```
Attention_weights @ V → [Batch x Num_Heads x Seq_len x Head_Dim]
```

---

## 🔹 Step 6: Concatenate Heads

1. Transpose back:
```
[Batch x Num_Heads x Seq_len x Head_Dim] → [Batch x Seq_len x Num_Heads x Head_Dim]
```

2. Flatten last two dims:
```
[Batch x Seq_len x Num_Heads × Head_Dim] = [Batch x Seq_len x Embed_dim]
```

This operation = **concatenation of heads**.

---

## 🔹 Step 7: Final Projection

Pass through `out_proj` (`Linear(embed_dim → embed_dim)`):  
So each token now has context from **all heads combined**.
```


In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.dropout = dropout
        
        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"
        
        # create pointwise projection layers for queries, keys, values
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        
        # output projection layer
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        # dropout layer
        self.dropout_layer = nn.Dropout(dropout)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.shape  # x: (4, 64, 128)
        
        # Step 1: Project input to queries, keys, values
        query = self.query_proj(x)  # (4, 64, 128) -> (4, 64, 128)
        key = self.key_proj(x)      # (4, 64, 128) -> (4, 64, 128)
        value = self.value_proj(x)  # (4, 64, 128) -> (4, 64, 128)
        
        # Step 2: Reshape for multi-head attention
        # Split embed_dim into (num_heads, head_dim)
        query = query.view(batch_size, seq_len, self.num_heads, self.head_dim)  # (4, 64, 8, 16)
        key = key.view(batch_size, seq_len, self.num_heads, self.head_dim)      # (4, 64, 8, 16)
        value = value.view(batch_size, seq_len, self.num_heads, self.head_dim)  # (4, 64, 8, 16)
        
        # Step 3: Transpose to get heads as separate batch dimension
        # Move num_heads dimension to position 1 for parallel processing
        query = query.transpose(1, 2)  # (4, 64, 8, 16) -> (4, 8, 64, 16)
        key = key.transpose(1, 2)      # (4, 64, 8, 16) -> (4, 8, 64, 16)
        value = value.transpose(1, 2)  # (4, 64, 8, 16) -> (4, 8, 64, 16)
        
        # Step 4: Compute attention scores for all heads simultaneously
        # Each head operates on head_dim=16 dimensions instead of full embed_dim=128
        similarity = (query @ key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        # (4, 8, 64, 16) @ (4, 8, 16, 64) -> (4, 8, 64, 64)
        # Scale by sqrt(head_dim) = sqrt(16) = 4
        
        # Step 5: Apply softmax to get attention weights
        attention_weights = F.softmax(similarity, dim=-1)  # (4, 8, 64, 64)
        
        # Step 6: Apply dropout to attention weights
        attention_weights = self.dropout_layer(attention_weights)
        
        # Step 7: Compute context vectors for each head
        context_vectors = attention_weights @ value  # (4, 8, 64, 64) @ (4, 8, 64, 16) -> (4, 8, 64, 16)
        
        # Step 8: Concatenate heads back together
        # First transpose back: (4, 8, 64, 16) -> (4, 64, 8, 16)
        context_vectors = context_vectors.transpose(1, 2)
        # Then reshape to merge heads: (4, 64, 8, 16) -> (4, 64, 128)
        context_vectors = context_vectors.contiguous().view(batch_size, seq_len, -1)
        
        # Step 9: Apply final output projection
        output = self.out_proj(context_vectors)  # (4, 64, 128) -> (4, 64, 128)
        
        return output

In [8]:
embed_dim = 128
num_heads = 8
multi_head_attention = MultiHeadAttention(embed_dim, num_heads)
context_vectors = multi_head_attention(x)
print("Context vectors shape with multi-head attention:", context_vectors.shape)

Context vectors shape with multi-head attention: torch.Size([4, 64, 128])


In [9]:
seq_len = 64
x = torch.randn(batch_size, seq_len, embed_dim)
print("Input shape:", x.shape)
output = multi_head_attention(x)
print("Output shape after multi-head attention:", output.shape)

Input shape: torch.Size([4, 64, 128])
Output shape after multi-head attention: torch.Size([4, 64, 128])


### some takeway

#### 🔢 Linear Layers on Multi-Dimensional Tensors (PyTorch)

- `nn.Linear(in_features, out_features)` expects input:  
  `[Batch x in_features] → [Batch x out_features]`

- If input = `[Batch x D1 x D2 x in_features]`  
  → Output = `[Batch x D1 x D2 x out_features]`

- ✅ PyTorch applies `nn.Linear` **only on the last dimension**
- ✅ All other dims are treated as **batch dimensions**

In [10]:
fc = nn.Linear(10, 30)

tensor_1 = torch.randn(4, 5, 10)
tensor_1_output = fc(tensor_1)
print("Output shape after linear layer on multi-dimensional tensor:", tensor_1_output.shape)

# Linear Layers on MultiDimensional Tensors
tensor_2 = torch.randn(5, 1, 2, 3, 4, 10)
tensor_2_output = fc(tensor_2)
print("Output shape after linear layer on multi-dimensional tensor:", tensor_2_output.shape)

Output shape after linear layer on multi-dimensional tensor: torch.Size([4, 5, 30])
Output shape after linear layer on multi-dimensional tensor: torch.Size([5, 1, 2, 3, 4, 30])


In [11]:
tensor = torch.randn(1, 8, 9)
fc = nn.Linear(9, 9)
output_tensor = fc(tensor)
print("Output tensor shape after linear layer:", output_tensor.shape)

q_head_1, q_head_2, q_head_3 = torch.chunk(output_tensor, 3, dim=-1) # Split into 3 heads
print("Shapes of query heads:", q_head_1.shape, q_head_2.shape, q_head_3.shape)

Output tensor shape after linear layer: torch.Size([1, 8, 9])
Shapes of query heads: torch.Size([1, 8, 3]) torch.Size([1, 8, 3]) torch.Size([1, 8, 3])


In [12]:
class SelfAttentionEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, attn_p=0.1, proj_p=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.attn_p = attn_p
        self.proj_p = proj_p

        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"

        # Pointwise projections for Q, K, V
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)

        # Output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        # Dropout layers
        self.attn_drop = nn.Dropout(attn_p)
        self.proj_drop = nn.Dropout(proj_p)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape  # x: (B, T, C)

        # Project to Q, K, V
        query = self.query_proj(x)
        key = self.key_proj(x)
        value = self.value_proj(x)

        # Reshape: [B, T, C] -> [B, T, H, D]
        query = query.view(batch_size, seq_len, self.num_heads, self.head_dim)
        key = key.view(batch_size, seq_len, self.num_heads, self.head_dim)
        value = value.view(batch_size, seq_len, self.num_heads, self.head_dim)

        # Transpose for attention: [B, H, T, D]
        query = query.transpose(1, 2)
        key = key.transpose(1, 2)
        value = value.transpose(1, 2)

        # Scaled dot-product attention
        similarity = (query @ key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attention_weights = F.softmax(similarity, dim=-1)
        attention_weights = self.attn_drop(attention_weights)

        # Context vectors
        context = attention_weights @ value  # [B, H, T, D]

        # Recombine heads: [B, T, H, D] -> [B, T, C]
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        # Final projection
        output = self.out_proj(context)
        output = self.proj_drop(output)

        return output

In [13]:
embed_dim = 9
num_heads = 3
seq_len = 8
a = SelfAttentionEncoder(embed_dim, num_heads)

### Create a random tensor in the shape (Batch x Seq Len x Embed Dim) ###
rand = torch.randn(3,seq_len,embed_dim)

### Pass through MHA ###
output = a(rand)
print("Final Output:", output.shape)

Final Output: torch.Size([3, 8, 9])


## Padding

In [14]:
rand_attn = torch.randn(1, 6, 6)
attention_mask = torch.tensor([1, 1, 1, 1, 0, 0]).unsqueeze(0).bool()  # Add batch dimension
print("Attention mask shape:", attention_mask.shape)

### Add Extra Dimension for the (b x n x n) ###
### So unsqueeze mask to be (b x 1 x n) ###
attention_mask = attention_mask.unsqueeze(1)
print("Attention mask shape after unsqueeze:", attention_mask.shape)

print(attention_mask)
print(rand_attn.masked_fill(~attention_mask, float('-inf')))

Attention mask shape: torch.Size([1, 6])
Attention mask shape after unsqueeze: torch.Size([1, 1, 6])
tensor([[[ True,  True,  True,  True, False, False]]])
tensor([[[-1.1260, -1.2834,  0.5358, -1.2702,    -inf,    -inf],
         [ 0.8232, -0.2840,  0.2091, -1.2259,    -inf,    -inf],
         [-0.7836, -0.0249,  0.5464, -0.3646,    -inf,    -inf],
         [-0.3225,  1.5512,  1.1823,  1.7917,    -inf,    -inf],
         [ 0.6686,  1.7344,  0.1981,  0.5730,    -inf,    -inf],
         [ 0.6711, -1.1842, -1.1230,  0.4570,    -inf,    -inf]]])


In [15]:
## Create an example attention matrix (b x h x n x n) ###
rand_attn = torch.rand(1,2,6,6) # I have 2 heads here!

### Create Attention Mask in the shape (b x n) ###
attention_mask = torch.tensor([1,1,1,1,0,0]).unsqueeze(0).bool()

### Add Two Extra Dimension for the (b x h x n x n) ###
### So unsqueeze mask to be (b x 1 x 1 x n) ###
attention_mask = attention_mask.unsqueeze(1).unsqueeze(1)

### Unsqueezed with dummy broadcast dimension ###
print(attention_mask)
print(rand_attn.masked_fill_(~attention_mask, float("-inf")))

tensor([[[[ True,  True,  True,  True, False, False]]]])
tensor([[[[0.7885, 0.7307, 0.6539, 0.3927,   -inf,   -inf],
          [0.8092, 0.6950, 0.8019, 0.4760,   -inf,   -inf],
          [0.4642, 0.6451, 0.3733, 0.4665,   -inf,   -inf],
          [0.3428, 0.2807, 0.8906, 0.5879,   -inf,   -inf],
          [0.6621, 0.5541, 0.6974, 0.7380,   -inf,   -inf],
          [0.1098, 0.4920, 0.2314, 0.5780,   -inf,   -inf]],

         [[0.0275, 0.1412, 0.9903, 0.7705,   -inf,   -inf],
          [0.4663, 0.4764, 0.7764, 0.4194,   -inf,   -inf],
          [0.1687, 0.7990, 0.2304, 0.5560,   -inf,   -inf],
          [0.5555, 0.1866, 0.7159, 0.8256,   -inf,   -inf],
          [0.3874, 0.4247, 0.1672, 0.8655,   -inf,   -inf],
          [0.6233, 0.8379, 0.5686, 0.1583,   -inf,   -inf]]]])


In [16]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, attn_p=0.1, proj_p=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.attn_p = attn_p
        self.proj_p = proj_p

        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"

        # Pointwise projections for Q, K, V
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)

        # Output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        # Dropout layers
        self.attn_drop = nn.Dropout(attn_p)
        self.proj_drop = nn.Dropout(proj_p)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len, _ = x.shape  # x: (B, T, C)

        # Project to Q, K, V
        query = self.query_proj(x)
        key = self.key_proj(x)
        value = self.value_proj(x)

        # Reshape: [B, T, C] -> [B, T, H, D]
        query = query.view(batch_size, seq_len, self.num_heads, self.head_dim)
        key = key.view(batch_size, seq_len, self.num_heads, self.head_dim)
        value = value.view(batch_size, seq_len, self.num_heads, self.head_dim)

        # Transpose for attention: [B, H, T, D]
        query = query.transpose(1, 2)
        key = key.transpose(1, 2)
        value = value.transpose(1, 2)

        # Scaled dot-product attention
        similarity = (query @ key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        # Apply attention mask if provided
        if attention_mask is not None:
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
            similarity = similarity.masked_fill(~attention_mask, float('-inf'))
            
        attention_weights = F.softmax(similarity, dim=-1)
        attention_weights = self.attn_drop(attention_weights)

        # Context vectors
        context = attention_weights @ value  # [B, H, T, D]

        # Recombine heads: [B, T, H, D] -> [B, T, C]
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        # Final projection
        output = self.out_proj(context)
        output = self.proj_drop(output)

        return output

In [17]:
### We will now have sequences of different lengths, identify the number of tokens in each sequence ###
seq_lens = [3,5,4]
embed_dim = 9
num_heads = 3
a = SelfAttention(embed_dim, num_heads)

### Create a random tensor in the shape (Batch x Seq Len x Embed Dim) ###
### This will be a tensor upto the max(seq_lens) ###
rand = torch.randn(len(seq_lens),max(seq_lens),embed_dim)

### Create Attention Mask from the seq_lens (shortest sequences padded to the longest ###
masks = torch.nn.utils.rnn.pad_sequence([torch.ones(l) for l in seq_lens], batch_first=True, padding_value=0).bool()
print("Attention Mask:")
print(masks)

### Pass through MHA ###
output = a(rand, attention_mask=masks)
print("Final Output:", output.shape)

Attention Mask:
tensor([[ True,  True,  True, False, False],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True,  True, False]])
Final Output: torch.Size([3, 5, 9])


## causal masking

In [18]:
seq_len = 8

# create a causal mask for a sequence of length 8
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
print("Causal Mask:")
print(causal_mask)

Causal Mask:
tensor([[ True, False, False, False, False, False, False, False],
        [ True,  True, False, False, False, False, False, False],
        [ True,  True,  True, False, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True,  True, False, False, False],
        [ True,  True,  True,  True,  True,  True, False, False],
        [ True,  True,  True,  True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True,  True,  True,  True]])


In [19]:
padding_mask = torch.tensor([1,1,1,1,0,0,0,0]).bool()  # Example padding mask
padding_mask = padding_mask.unsqueeze(0).repeat(seq_len, 1)  # Add extra dimensions for broadcasting
print("Padding Mask:")
print(padding_mask)

Padding Mask:
tensor([[ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False]])


In [20]:
seq_len = 8

### Create Causal Mask ###
ones = torch.ones((seq_len, seq_len))
causal_mask = torch.tril(ones).bool()

### Create Padding Mask ###
padding_mask = torch.tensor([1,1,1,1,0,0,0,0]).bool()
padding_mask = padding_mask.unsqueeze(0).repeat(seq_len,1)

### Combine Masks (set positions we dont want in our causal mask to False based on the padding mask) ###
causal_mask = causal_mask.masked_fill(~padding_mask, 0)
causal_mask

tensor([[ True, False, False, False, False, False, False, False],
        [ True,  True, False, False, False, False, False, False],
        [ True,  True,  True, False, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False]])

In [21]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, attn_p=0.1, proj_p=0.1, causal_mask=False):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.attn_p = attn_p
        self.proj_p = proj_p
        self.causal_mask = causal_mask  # Store as instance attribute

        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"

        # Pointwise projections for Q, K, V
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)

        # Output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        # Dropout layers
        self.attn_drop = nn.Dropout(attn_p)
        self.proj_drop = nn.Dropout(proj_p)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len, _ = x.shape  # x: (B, T, C)

        # Project to Q, K, V
        query = self.query_proj(x)
        key = self.key_proj(x)
        value = self.value_proj(x)

        # Reshape: [B, T, C] -> [B, T, H, D]
        query = query.view(batch_size, seq_len, self.num_heads, self.head_dim)
        key = key.view(batch_size, seq_len, self.num_heads, self.head_dim)
        value = value.view(batch_size, seq_len, self.num_heads, self.head_dim)

        # Transpose for attention: [B, H, T, D]
        query = query.transpose(1, 2)
        key = key.transpose(1, 2)
        value = value.transpose(1, 2)

        # Scaled dot-product attention
        similarity = (query @ key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        # Apply causal mask if enabled
        if self.causal_mask:
            # Create causal mask
            causal_attn_mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).bool()
            causal_attn_mask = causal_attn_mask.unsqueeze(0).unsqueeze(0)  # Add batch and head dimensions
            similarity = similarity.masked_fill(~causal_attn_mask, float('-inf'))
            
        # Apply attention mask if provided (for padding)
        if attention_mask is not None:
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)  # Add head and query dimensions
            similarity = similarity.masked_fill(~attention_mask, float('-inf'))
            
        attention_weights = F.softmax(similarity, dim=-1)
        attention_weights = self.attn_drop(attention_weights)

        # Context vectors
        context = attention_weights @ value  # [B, H, T, D]

        # Recombine heads: [B, T, H, D] -> [B, T, C]
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        # Final projection
        output = self.out_proj(context)
        output = self.proj_drop(output)

        return output

In [22]:
### We will now have sequences of different lengths, identify the number of tokens in each sequence ###
seq_lens = [3,5,4]
embed_dim = 9
num_heads = 3
a = SelfAttention(embed_dim, num_heads, causal_mask=True)

### Create a random tensor in the shape (Batch x Seq Len x Embed Dim) ###
### This will be a tensor upto the max(seq_lens) ###
rand = torch.randn(len(seq_lens),max(seq_lens),embed_dim)

### Create Attention Mask from the seq_lens (shortest sequences padded to the longest ###
masks = torch.nn.utils.rnn.pad_sequence([torch.ones(l) for l in seq_lens], batch_first=True, padding_value=0).bool()
print("Attention Mask:")
print(masks)

### Pass through MHA ###
output = a(rand, attention_mask=masks)
print("Final Output:", output.shape)

Attention Mask:
tensor([[ True,  True,  True, False, False],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True,  True, False]])
Final Output: torch.Size([3, 5, 9])
